In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import torch
# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.predictor import ShorelinePredictor

In [ ]:
from src.data_processing.dataset_loader import CoastData
from src.models.data_management.data_loader import DataLoaderManager

import cv2

In [ ]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [ ]:
model_weight_path = os.path.abspath(os.path.join(os.getcwd(), "../artifacts/", "2025-10-02-10-54-16_2_classes_256x1024_b24/models/best_model.pth"))

model = "DeepLabV3"
num_classes = 2

predictor = ShorelinePredictor(model, model_weight_path, num_classes)

In [ ]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../data/processed_obliques_2_classes"))

coast_data = CoastData(data_path)
split = coast_data.split_data()

data = DataLoaderManager.load_data(split)

In [ ]:
index = 0
img_path = sorted(data["test"]['images'])[index]
print(img_path)

image = cv2.imread(img_path)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
# plt.axis('off')
plt.show()

print("Image shape:", image.shape)

crop = ((0,1500), (461,3500))  # ((y_min, y_max), (x_min, x_max))

output = predictor.predict_roi(img_path, crop_coords=crop, patch_size=(256, 1024), stride=(128, 512))

plt.imshow(output["predicted_image"])
plt.show()

plt.imshow(output["shoreline_mask"], cmap='gray')
plt.show()

print("New image shape:", output["predicted_image"].shape)
print("Shoreline coordinates:", output["shoreline_coords"])


# Prediction from SCLabels (rectified)

In [ ]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../data/SCLabels_v1.0.0"))

coast_data = CoastData(data_path, name="arenaldentem")
split = coast_data.split_data()

data = DataLoaderManager.load_data(split)

In [ ]:
model_weight_path = os.path.abspath(os.path.join(os.getcwd(), "../artifacts/", "2025-03-12-11-44-51_deeplab/models/best_model.pth"))

model = "DeepLabV3"
num_classes = 3

predictor = ShorelinePredictor(model, model_weight_path, num_classes)

In [ ]:
index = 3
img_path = sorted(data["test"]['images'])[index]
print(img_path)

image = cv2.imread(img_path)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
# plt.axis('off')
plt.show()

print("Image shape:", image.shape)

mask_path = sorted(data["test"]['masks'])[index]
output = predictor.predict_rectified_with_mask(img_path, mask_path=mask_path, patch_size=(256, 256), stride=(128, 128))

plt.imshow(output["predicted_image"])
plt.show()

plt.imshow(output["shoreline_mask"], cmap='gray')
plt.show()

plt.imshow(output["predicted_mask"], cmap='gray')
plt.show()

# print("New image shape:", output["predicted_image"].shape)
print("Shoreline coordinates:", output["shoreline_coords"])

# Prediction from SCLabels (oblique)

In [ ]:
import pandas as pd

In [ ]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../data/SCLabels_oblique_registered_v1.0.0/"))

coast_data = CoastData(data_path, name="arenaldentem")
data = coast_data.split_data(get_coords=True)

In [ ]:
model_weight_path = os.path.abspath(os.path.join(os.getcwd(), "../artifacts/", "2025-10-02-10-54-16_2_classes_256x1024_b24/models/best_model.pth"))

model = "DeepLabV3"
num_classes = 2

predictor = ShorelinePredictor(model, model_weight_path, num_classes)

In [ ]:
index = 1

split = "test"
img_path = data[split]['images'][index]
shoreline_coords = data[split]['masks'][index]
print(img_path)
print(data[split]['original_image_paths'][index])

In [ ]:
output = predictor.predict_oblique_with_coords(img_path, shoreline_coords=shoreline_coords, patch_size=(256, 1024), stride=(128, 512), for_matlab=True)

plt.imshow(output["predicted_image"])
plt.show()

plt.imshow(output["shoreline_mask"], cmap='gray')
plt.show()

plt.imshow(output["predicted_mask"], cmap='gray')
plt.show()

# print("New image shape:", output["predicted_image"].shape)
print("Shoreline coordinates:", output["shoreline_coords"])

In [ ]:
def export_csv_prediction(path_to_save, shoreline_coords):
    print(shoreline_coords)
    df = pd.DataFrame(shoreline_coords, columns=['u', 'v'])
    df.to_csv(path_to_save, index=False)
    print(f"Shoreline coordinates exported to {path_to_save}")

def modify_path(path):
    new_dir = path.replace("snap", "shoreline")
    new_dir = new_dir.replace(".jpg", ".csv")
    return new_dir


In [ ]:
new_path = modify_path(data[split]['original_image_paths'][index])
export_csv_prediction(new_path, output["shoreline_coords"])